In [ ]:

import math

# Model 03: Initial two-model PDVI formulation
# PDVI = 100 * sqrt(Model 01 disease confidence * expected severity)

SEVERITY_WEIGHTS = {
    "Low": 0.0,
    "Moderate": 1.0,
    "High": 2.0,
    "Severe": 3.0,
}

def calculate_pdvi(model01_disease, model01_confidence, model02_severity_probabilities):
    """
    Inputs:
      model01_disease: predicted disease label from Model 01
      model01_confidence: confidence for that predicted class, from 0 to 1
      model02_severity_probabilities: dict such as
          {"Low": 0.05, "Moderate": 0.25, "High": 0.10, "Severe": 0.60}

    Returns:
      A dictionary containing the PDVI score and calculation details.
    """

    disease = str(model01_disease).strip().lower().replace(" ", "_")

    if not 0 <= model01_confidence <= 1:
        raise ValueError("Model 01 confidence must be between 0 and 1.")

    # Healthy leaves do not have a disease-severity category.
    if disease == "healthy":
        return {
            "model01_disease": model01_disease,
            "model01_confidence": float(model01_confidence),
            "expected_severity": None,
            "pdvi_score": 0.0,
            "status": "Provisional zero: Model 01 predicted Healthy",
            "warning": "Requires field validation; not proof of zero plantation risk."
        }

    required = set(SEVERITY_WEIGHTS)
    supplied = set(model02_severity_probabilities)

    if supplied != required:
        raise ValueError(
            f"Model 02 probabilities must contain exactly: {sorted(required)}"
        )

    probabilities = {
        label: float(model02_severity_probabilities[label])
        for label in SEVERITY_WEIGHTS
    }

    if any(not math.isfinite(p) or p < 0 or p > 1
           for p in probabilities.values()):
        raise ValueError("Severity probabilities must be finite values from 0 to 1.")

    total_probability = sum(probabilities.values())

    if not math.isclose(total_probability, 1.0, abs_tol=0.01):
        raise ValueError(
            f"Severity probabilities must sum to approximately 1.0; got {total_probability:.4f}"
        )

    # Expected ordinal severity, normalized to 0-1.
    expected_severity = sum(
        probabilities[label] * weight
        for label, weight in SEVERITY_WEIGHTS.items()
    ) / 3.0

    # Proposed nonlinear fusion formula.
    pdvi = 100.0 * math.sqrt(model01_confidence * expected_severity)

    return {
        "model01_disease": model01_disease,
        "model01_confidence": round(float(model01_confidence), 4),
        "model02_severity_probabilities": probabilities,
        "expected_severity": round(expected_severity, 4),
        "pdvi_score": round(min(100.0, max(0.0, pdvi)), 2),
        "status": "Calculated — provisional research index",
        "warning": "Not a calibrated disease probability or validated field-risk score."
    }


# Verification example ONLY — these are illustrative values, not model predictions.
example = calculate_pdvi(
    model01_disease="Blister_Blight",
    model01_confidence=0.90,
    model02_severity_probabilities={
        "Low": 0.05,
        "Moderate": 0.25,
        "High": 0.10,
        "Severe": 0.60
    }
)

print("MODEL 03 — PDVI FORMULA TEST")
print("-" * 40)
for key, value in example.items():
    print(f"{key}: {value}")

assert 0.0 <= example["pdvi_score"] <= 100.0
assert example["expected_severity"] == 0.75
assert example["pdvi_score"] == 82.16

print("\nAll formula verification checks passed.")


MODEL 03 — PDVI FORMULA TEST
----------------------------------------
model01_disease: Blister_Blight
model01_confidence: 0.9
model02_severity_probabilities: {'Low': 0.05, 'Moderate': 0.25, 'High': 0.1, 'Severe': 0.6}
expected_severity: 0.75
pdvi_score: 82.16
status: Calculated — provisional research index

All formula verification checks passed.


In [ ]:
from pathlib import Path

ARTIFACTS = {
    "Model 01": "/content/gdrive/MyDrive/Tea_Disease_Research/model_outputs/best_tea_disease_efficientnetb0_finetuned.keras",
    "Model 01 class names": "/content/gdrive/MyDrive/Tea_Disease_Research/model_outputs/class_names.json",
    "Original Model 02": "/content/gdrive/MyDrive/Disease_Severity_And_Weather_Dataset/Model02_Final/Model02_Final_Model.joblib",
    "Model 02 severity encoder": "/content/gdrive/MyDrive/Disease_Severity_And_Weather_Dataset/Model02_Final/Model02_LabelEncoder.joblib",
    "Model 02 feature list": "/content/gdrive/MyDrive/Disease_Severity_And_Weather_Dataset/Model02_Final/Model02_Feature_List.json",
    "Environmental dataset": "/content/gdrive/MyDrive/Disease_Severity_And_Weather_Dataset/Member01_Model02_Environmental_IoT_Final.csv",
}

print("MODEL 03 — ARTIFACT CHECK")
print("-" * 45)

for name, file_path in ARTIFACTS.items():
    path = Path(file_path)
    print(f"{name}: {'FOUND' if path.is_file() else 'MISSING'}")
    if not path.is_file():
        print(f"  Expected path: {file_path}")

MODEL 03 — ARTIFACT CHECK
---------------------------------------------
Model 01: MISSING
  Expected path: /content/gdrive/MyDrive/Tea_Disease_Research/model_outputs/best_tea_disease_efficientnetb0_finetuned.keras
Model 01 class names: MISSING
  Expected path: /content/gdrive/MyDrive/Tea_Disease_Research/model_outputs/class_names.json
Original Model 02: MISSING
  Expected path: /content/gdrive/MyDrive/Disease_Severity_And_Weather_Dataset/Model02_Final/Model02_Final_Model.joblib
Model 02 severity encoder: MISSING
  Expected path: /content/gdrive/MyDrive/Disease_Severity_And_Weather_Dataset/Model02_Final/Model02_LabelEncoder.joblib
Model 02 feature list: MISSING
  Expected path: /content/gdrive/MyDrive/Disease_Severity_And_Weather_Dataset/Model02_Final/Model02_Feature_List.json
Environmental dataset: MISSING
  Expected path: /content/gdrive/MyDrive/Disease_Severity_And_Weather_Dataset/Member01_Model02_Environmental_IoT_Final.csv


In [ ]:

from google.colab import drive
from pathlib import Path

# Mount Google Drive
drive.mount("/content/drive")

# Search for the required files inside MyDrive
root = Path("/content/drive/MyDrive")

expected_names = {
    "best_tea_disease_efficientnetb0_finetuned.keras",
    "class_names.json",
    "Model02_Final_Model.joblib",
    "Model02_LabelEncoder.joblib",
    "Model02_Feature_List.json",
    "Member01_Model02_Environmental_IoT_Final.csv",
}

print("\nMODEL 03 — DRIVE FILE SEARCH")
print("-" * 50)

found = {}

for path in root.rglob("*"):
    if path.is_file() and path.name in expected_names:
        found[path.name] = str(path)
        print(f"FOUND: {path.name}")
        print(f"PATH:  {path}")

print("\n" + "-" * 50)

missing = expected_names - set(found)

if missing:
    print("FILES NOT FOUND:")
    for name in sorted(missing):
        print(f"- {name}")
    print("\nCheck whether the files are stored in another Drive account,")
    print("another folder, or have different filenames.")
else:
    print("All six required files were found.")


Mounted at /content/drive

MODEL 03 — DRIVE FILE SEARCH
--------------------------------------------------
FOUND: Member01_Model02_Environmental_IoT_Final.csv
PATH:  /content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/Member01_Model02_Environmental_IoT_Final.csv
FOUND: Model02_LabelEncoder.joblib
PATH:  /content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/Model02_Final/Model02_LabelEncoder.joblib
FOUND: Model02_Feature_List.json
PATH:  /content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/Model02_Final/Model02_Feature_List.json
FOUND: Model02_Final_Model.joblib
PATH:  /content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/Model02_Final/Model02_Final_Model.joblib
FOUND: class_names.json
PATH:  /content/drive/MyDrive/Tea_Disease_Research/model_outputs/class_names.json
FOUND: best_tea_disease_efficientnetb0_finetuned.keras
PATH:  /content/drive/MyDrive/Tea_Disease_Research/model_outputs/best_tea_disease_efficientnetb0_finetuned.keras

----------------------------

In [ ]:

from pathlib import Path
import json
import joblib
import pandas as pd
import tensorflow as tf

# Exact paths found in your Google Drive
BASE = Path("/content/drive/MyDrive")

MODEL01_PATH = BASE / "Tea_Disease_Research/model_outputs/best_tea_disease_efficientnetb0_finetuned.keras"
CLASS_NAMES_PATH = BASE / "Tea_Disease_Research/model_outputs/class_names.json"

MODEL02_DIR = BASE / "Disease_Severity_And_Weather_Dataset/Model02_Final"
MODEL02_PATH = MODEL02_DIR / "Model02_Final_Model.joblib"
ENCODER_PATH = MODEL02_DIR / "Model02_LabelEncoder.joblib"
FEATURES_PATH = MODEL02_DIR / "Model02_Feature_List.json"

ENV_PATH = BASE / "Disease_Severity_And_Weather_Dataset/Member01_Model02_Environmental_IoT_Final.csv"

print("MODEL 03 — ARTIFACT LOADING")
print("=" * 55)

# Load Model 01 without compiling or changing it
model01 = tf.keras.models.load_model(
    str(MODEL01_PATH),
    compile=False
)

with open(CLASS_NAMES_PATH, "r") as f:
    class_names = json.load(f)

# Load Original Model 02 and its supporting artifacts
model02 = joblib.load(MODEL02_PATH)
severity_encoder = joblib.load(ENCODER_PATH)

with open(FEATURES_PATH, "r") as f:
    model02_features = json.load(f)

environment_df = pd.read_csv(ENV_PATH)

print("\nMODEL 01")
print("Input shape:", model01.input_shape)
print("Output shape:", model01.output_shape)
print("Class names:", class_names)

print("\nORIGINAL MODEL 02")
print("Model type:", type(model02).__name__)
print("Pipeline steps:", list(model02.named_steps.keys())
      if hasattr(model02, "named_steps") else "Not a pipeline")
print("Encoded model classes:", model02.classes_)
print("Encoder classes:", severity_encoder.classes_)
print("Required feature count:", len(model02_features))
print("Required features:", model02_features)
print("Supports predict_proba:", hasattr(model02, "predict_proba"))

print("\nENVIRONMENTAL DATASET")
print("Shape:", environment_df.shape)
print("Columns:", environment_df.columns.tolist())

for col in ["estate_id", "estate_name"]:
    if col in environment_df.columns:
        print(f"{col} values:", sorted(
            environment_df[col].dropna().astype(str).unique().tolist()
        ))

print("\nVERIFICATION")
print("Model 01 loaded:", model01 is not None)
print("Model 02 loaded:", model02 is not None)
print("Severity encoder loaded:", severity_encoder is not None)
print("Feature list loaded:", len(model02_features) > 0)
print("Environmental data loaded:", not environment_df.empty)

print("\nNo model training or artifact saving was performed.")


MODEL 03 — ARTIFACT LOADING

MODEL 01
Input shape: (None, 224, 224, 3)
Output shape: (None, 4)
Class names: ['Blister_Blight', 'Grey_Blight', 'Red_Rust', 'Healthy']

ORIGINAL MODEL 02
Model type: Pipeline
Pipeline steps: ['preprocessor', 'model']
Encoded model classes: [0 1 2 3]
Encoder classes: ['High' 'Low' 'Moderate' 'Severe']
Required feature count: 19
Required features: ['temperature_c', 'relative_humidity_pct', 'soil_moisture_pct', 'leaf_wetness', 'rainfall_mm', 'light_intensity_lux', 'wind_speed', 'temperature_c_lag1', 'temperature_c_lag3', 'temperature_c_lag7', 'relative_humidity_pct_lag7', 'rainfall_mm_lag7', 'soil_moisture_pct_lag3', 'soil_moisture_pct_prev3d_mean', 'relative_humidity_pct_prev7d_mean', 'rainfall_mm_prev7d_mean', 'month', 'disease_class', 'estate_id']
Supports predict_proba: True

ENVIRONMENTAL DATASET
Shape: (9130, 10)
Columns: ['timestamp', 'estate_id', 'estate_name', 'temperature_c', 'relative_humidity_pct', 'soil_moisture_pct', 'leaf_wetness', 'rainfall_mm

In [ ]:

import pandas as pd
import numpy as np

df = environment_df.copy()
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df.sort_values(["estate_id", "timestamp"]).reset_index(drop=True)

sensor_cols = [
    "temperature_c",
    "relative_humidity_pct",
    "soil_moisture_pct",
    "leaf_wetness",
    "rainfall_mm",
    "light_intensity_lux",
    "wind_speed",
]

print("MODEL 03 — ENVIRONMENTAL DATA AUDIT")
print("=" * 60)

print("\n1. DATE COVERAGE")
print("Invalid timestamps:", df["timestamp"].isna().sum())
print("Date range:", df["timestamp"].min(), "to", df["timestamp"].max())

print("\n2. RECORDS PER ESTATE")
print(df.groupby(["estate_id", "estate_name"]).agg(
    rows=("timestamp", "size"),
    first_date=("timestamp", "min"),
    last_date=("timestamp", "max"),
    unique_dates=("timestamp", "nunique"),
).to_string())

print("\n3. MISSING VALUES")
print(df[sensor_cols].isna().sum().to_string())

print("\n4. SENSOR VALUE RANGES")
print(df[sensor_cols].agg(["min", "median", "max"]).T.to_string())

print("\n5. DUPLICATE ESTATE-DATE RECORDS")
valid_dates = df.dropna(subset=["timestamp"])
print(
    valid_dates.duplicated(["estate_id", "timestamp"]).sum()
)

print("\n6. LAG / ROLLING FEATURE READINESS")
for estate_id, group in df.groupby("estate_id"):
    group = group.sort_values("timestamp")
    dates = group["timestamp"].dt.normalize()
    gaps = dates.diff().dt.days

    print(
        f"{estate_id}: rows={len(group)}, "
        f"consecutive daily intervals={(gaps == 1).sum()}, "
        f"date gaps={(gaps > 1).sum()}"
    )

print("\n7. ESTATE-ID WARNING")
print("Environmental IDs:", sorted(df["estate_id"].dropna().unique()))
print("Model 02 expects estate IDs from its own training data.")
print("Do not translate E01-E05 to EST001-EST005 without verified evidence.")

print("\nAudit complete. No files or models were modified.")


MODEL 03 — ENVIRONMENTAL DATA AUDIT

1. DATE COVERAGE
Invalid timestamps: 5530
Date range: 2021-01-01 00:00:00 to 2025-12-12 00:00:00

2. RECORDS PER ESTATE
                                   rows first_date  last_date  unique_dates
estate_id estate_name                                                      
E01       Nuwara Eliya Tea Estate  1826 2021-01-01 2025-12-12           720
E02       Badulla Tea Estate       1826 2021-01-01 2025-12-12           720
E03       Kandy Tea Estate         1826 2021-01-01 2025-12-12           720
E04       Ratnapura Tea Estate     1826 2021-01-01 2025-12-12           720
E05       Galle Tea Estate         1826 2021-01-01 2025-12-12           720

3. MISSING VALUES
temperature_c            0
relative_humidity_pct    0
soil_moisture_pct        0
leaf_wetness             0
rainfall_mm              0
light_intensity_lux      0
wind_speed               0

4. SENSOR VALUE RANGES
                           min     median       max
temperature_c            11

In [ ]:

import pandas as pd
from pathlib import Path

ENV_PATH = Path(
    "/content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/"
    "Member01_Model02_Environmental_IoT_Final.csv"
)

# Read the original timestamp strings without automatic date conversion
df_fix = pd.read_csv(ENV_PATH, dtype={"timestamp": "string"})
raw = df_fix["timestamp"].str.strip()

print("MODEL 03 — TIMESTAMP FORMAT FIX")
print("=" * 50)

print("\n1. Raw timestamp examples:")
print(raw.dropna().drop_duplicates().head(15).to_list())

# Try common date formats independently
formats = [
    "%Y-%m-%d",
    "%Y-%m-%d %H:%M:%S",
    "%d/%m/%Y",
    "%d/%m/%Y %H:%M:%S",
    "%m/%d/%Y",
    "%m/%d/%Y %H:%M:%S",
    "%d-%m-%Y",
    "%d-%m-%Y %H:%M:%S",
]

candidates = {}

for fmt in formats:
    parsed = pd.to_datetime(raw, format=fmt, errors="coerce")
    candidates[fmt] = parsed
    print(f"{fmt}: {parsed.notna().sum()}/{len(raw)} parsed")

# Choose the format that parses the most rows
best_format = max(candidates, key=lambda fmt: candidates[fmt].notna().sum())
best_parsed = candidates[best_format]

print("\nBest format:", best_format)
print("Rows parsed:", best_parsed.notna().sum())
print("Rows still invalid:", best_parsed.isna().sum())

# If a single format cannot parse everything, test mixed-format parsing
if best_parsed.isna().any():
    try:
        mixed_parsed = pd.to_datetime(
            raw, format="mixed", errors="coerce", dayfirst=True
        )
        if mixed_parsed.notna().sum() > best_parsed.notna().sum():
            best_parsed = mixed_parsed
            print("Mixed-format parsing improved the result.")
    except (TypeError, ValueError):
        pass

df_fix["timestamp"] = best_parsed

print("\n2. Corrected date coverage:")
print("Invalid timestamps:", df_fix["timestamp"].isna().sum())
print("First date:", df_fix["timestamp"].min())
print("Last date:", df_fix["timestamp"].max())

print("\n3. Unique dates per estate:")
print(
    df_fix.dropna(subset=["timestamp"])
    .assign(date=lambda x: x["timestamp"].dt.normalize())
    .groupby("estate_id")["date"]
    .nunique()
    .to_string()
)

print("\n4. Examples still invalid:")
print(raw[df_fix["timestamp"].isna()].head(20).to_list())

print("\nOriginal CSV remains unchanged.")


MODEL 03 — TIMESTAMP FORMAT FIX

1. Raw timestamp examples:
['1/01/2021', '2/01/2021', '3/01/2021', '4/01/2021', '5/01/2021', '6/01/2021', '7/01/2021', '8/01/2021', '9/01/2021', '10/01/2021', '11/01/2021', '12/01/2021', '13/01/2021', '14/01/2021', '15/01/2021']
%Y-%m-%d: 0/9130 parsed
%Y-%m-%d %H:%M:%S: 0/9130 parsed
%d/%m/%Y: 9130/9130 parsed
%d/%m/%Y %H:%M:%S: 0/9130 parsed
%m/%d/%Y: 3600/9130 parsed
%m/%d/%Y %H:%M:%S: 0/9130 parsed
%d-%m-%Y: 0/9130 parsed
%d-%m-%Y %H:%M:%S: 0/9130 parsed

Best format: %d/%m/%Y
Rows parsed: 9130
Rows still invalid: 0

2. Corrected date coverage:
Invalid timestamps: 0
First date: 2021-01-01 00:00:00
Last date: 2025-12-31 00:00:00

3. Unique dates per estate:
estate_id
E01    1826
E02    1826
E03    1826
E04    1826
E05    1826

4. Examples still invalid:
[]

Original CSV remains unchanged.


In [ ]:

# Continue using df_fix from the timestamp correction cell
df_history = df_fix.copy()

df_history["timestamp"] = pd.to_datetime(
    df_history["timestamp"], errors="raise"
)
df_history["date"] = df_history["timestamp"].dt.normalize()

sensor_cols = [
    "temperature_c",
    "relative_humidity_pct",
    "soil_moisture_pct",
    "leaf_wetness",
    "rainfall_mm",
    "light_intensity_lux",
    "wind_speed",
]

df_history = df_history.sort_values(
    ["estate_id", "date"]
).reset_index(drop=True)

# Confirm each estate has exactly one row per calendar date
print("MODEL 03 — HISTORICAL FEATURE CHECK")
print("=" * 55)

for estate_id, group in df_history.groupby("estate_id"):
    expected_dates = pd.date_range(
        group["date"].min(),
        group["date"].max(),
        freq="D"
    )
    actual_dates = pd.DatetimeIndex(group["date"].drop_duplicates())
    missing_dates = expected_dates.difference(actual_dates)

    print(
        f"{estate_id}: rows={len(group)}, "
        f"missing calendar dates={len(missing_dates)}"
    )

# Generate date-aware lag and rolling features per estate
for col in [
    "temperature_c",
    "relative_humidity_pct",
    "rainfall_mm",
    "soil_moisture_pct",
]:
    for lag in [1, 3, 7]:
        if col == "temperature_c" or (
            col == "relative_humidity_pct" and lag == 7
        ) or (
            col == "rainfall_mm" and lag == 7
        ) or (
            col == "soil_moisture_pct" and lag == 3
        ):
            feature_name = f"{col}_lag{lag}"
            df_history[feature_name] = (
                df_history.groupby("estate_id")[col].shift(lag)
            )

# Rolling means use prior days only, excluding the current day
df_history["soil_moisture_pct_prev3d_mean"] = (
    df_history.groupby("estate_id")["soil_moisture_pct"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=3).mean())
)

df_history["relative_humidity_pct_prev7d_mean"] = (
    df_history.groupby("estate_id")["relative_humidity_pct"]
    .transform(lambda s: s.shift(1).rolling(7, min_periods=7).mean())
)

df_history["rainfall_mm_prev7d_mean"] = (
    df_history.groupby("estate_id")["rainfall_mm"]
    .transform(lambda s: s.shift(1).rolling(7, min_periods=7).mean())
)

df_history["month"] = df_history["date"].dt.month

required_history_features = [
    "temperature_c_lag1",
    "temperature_c_lag3",
    "temperature_c_lag7",
    "relative_humidity_pct_lag7",
    "rainfall_mm_lag7",
    "soil_moisture_pct_lag3",
    "soil_moisture_pct_prev3d_mean",
    "relative_humidity_pct_prev7d_mean",
    "rainfall_mm_prev7d_mean",
]

print("\nHistorical feature missing counts:")
print(df_history[required_history_features].isna().sum().to_string())

print("\nRows with complete historical features:",
      int(df_history[required_history_features].notna().all(axis=1).sum()))

print("\nFeature calculation complete. No files saved.")


MODEL 03 — HISTORICAL FEATURE CHECK
E01: rows=1826, missing calendar dates=0
E02: rows=1826, missing calendar dates=0
E03: rows=1826, missing calendar dates=0
E04: rows=1826, missing calendar dates=0
E05: rows=1826, missing calendar dates=0

Historical feature missing counts:
temperature_c_lag1                    5
temperature_c_lag3                   15
temperature_c_lag7                   35
relative_humidity_pct_lag7           35
rainfall_mm_lag7                     35
soil_moisture_pct_lag3               15
soil_moisture_pct_prev3d_mean        15
relative_humidity_pct_prev7d_mean    35
rainfall_mm_prev7d_mean              35

Rows with complete historical features: 9095

Feature calculation complete. No files saved.


In [ ]:

import pandas as pd
import joblib

DATA_DIR = (
    "/content/drive/MyDrive/"
    "Disease_Severity_And_Weather_Dataset"
)

# Inspect Model 02's actual training pipeline categories
preprocessor = model02.named_steps["preprocessor"]

print("MODEL 02 — INPUT COMPATIBILITY CHECK")
print("=" * 55)

print("\n1. Model 02 encoder classes:")
print(severity_encoder.classes_)

print("\n2. Environmental estate IDs and names:")
print(
    df_history[["estate_id", "estate_name"]]
    .drop_duplicates()
    .sort_values("estate_id")
    .to_string(index=False)
)

print("\n3. Model 02 categorical preprocessing:")
for name, transformer, columns in preprocessor.transformers_:
    if name == "remainder" or transformer == "drop":
        continue

    if isinstance(columns, (list, tuple)):
        categorical = [
            col for col in columns
            if col in ["estate_id", "disease_class"]
        ]
        if categorical:
            print(f"Transformer: {name}")
            print("Categorical columns:", categorical)
            print("Transformer:", transformer)

            if hasattr(transformer, "categories_"):
                for col, categories in zip(
                    categorical, transformer.categories_
                ):
                    print(f"{col} categories:", categories)

print("\n4. Model 02 required categorical inputs:")
print("disease_class required:", "disease_class" in model02_features)
print("estate_id required:", "estate_id" in model02_features)

print("\nCompatibility inspection complete.")


MODEL 02 — INPUT COMPATIBILITY CHECK

1. Model 02 encoder classes:
['High' 'Low' 'Moderate' 'Severe']

2. Environmental estate IDs and names:
estate_id             estate_name
      E01 Nuwara Eliya Tea Estate
      E02      Badulla Tea Estate
      E03        Kandy Tea Estate
      E04    Ratnapura Tea Estate
      E05        Galle Tea Estate

3. Model 02 categorical preprocessing:
Transformer: categorical
Categorical columns: ['disease_class', 'estate_id']
Transformer: OneHotEncoder(handle_unknown='ignore')
disease_class categories: ['Blister_Blight' 'Grey_Blight' 'Red_Rust']
estate_id categories: ['EST001' 'EST002' 'EST003' 'EST004' 'EST005']

4. Model 02 required categorical inputs:
disease_class required: True
estate_id required: True

Compatibility inspection complete.


In [ ]:

from pathlib import Path

ROOT = Path(
    "/content/drive/MyDrive/Disease_Severity_And_Weather_Dataset"
)

print("MODEL 03 — ESTATE MAPPING INVESTIGATION")
print("=" * 55)

print("\n1. Relevant CSV files:")
for path in sorted(ROOT.rglob("*.csv")):
    print(path)

print("\n2. Files potentially containing estate mappings:")
keywords = ("estate", "mapping", "model02", "training", "dataset")

for path in sorted(ROOT.rglob("*")):
    if not path.is_file():
        continue

    name = path.name.lower()
    if any(word in name for word in keywords):
        if path.suffix.lower() in {
            ".csv", ".json", ".txt", ".xlsx", ".md"
        }:
            print(path)

print("\nSearch complete. No files were changed.")


MODEL 03 — ESTATE MAPPING INVESTIGATION

1. Relevant CSV files:
/content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/Member01_Model02_Disease_Occurrence_Severity_Final.csv
/content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/Member01_Model02_Environmental_IoT_Final.csv
/content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/Model02_Aligned_V2/Model02_Aligned_V2_Training_Data.csv
/content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/Model02_XGBoost_Results/model02_test_predictions.csv
/content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/Severity_Training_Data.csv

2. Files potentially containing estate mappings:
/content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/Member01_Model02_Disease_Occurrence_Severity_Final.csv
/content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/Member01_Model02_Environmental_IoT_Final.csv
/content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/Model02_Aligned_V2/Model02_Aligned_V2_Evaluation_Report.json
/content/

In [ ]:

# MODEL 03 — PROVISIONAL ESTATE ID MAPPING
# For prototype testing only; not verified training-data correspondence.

provisional_estate_mapping = {
    "E01": "EST001",  # Nuwara Eliya
    "E02": "EST002",  # Badulla
    "E03": "EST003",  # Kandy
    "E04": "EST004",  # Ratnapura
    "E05": "EST005",  # Galle
}

df_history["model02_estate_id"] = (
    df_history["estate_id"].map(provisional_estate_mapping)
)

print("MODEL 03 — PROVISIONAL MAPPING")
print("=" * 50)

print(
    df_history[
        ["estate_id", "estate_name", "model02_estate_id"]
    ]
    .drop_duplicates()
    .sort_values("estate_id")
    .to_string(index=False)
)

print("\nUnmapped rows:", df_history["model02_estate_id"].isna().sum())
print("Mapping status: PROVISIONAL — NOT VERIFIED")


MODEL 03 — PROVISIONAL MAPPING
estate_id             estate_name model02_estate_id
      E01 Nuwara Eliya Tea Estate            EST001
      E02      Badulla Tea Estate            EST002
      E03        Kandy Tea Estate            EST003
      E04    Ratnapura Tea Estate            EST004
      E05        Galle Tea Estate            EST005

Unmapped rows: 0
Mapping status: PROVISIONAL — NOT VERIFIED


In [ ]:

import pandas as pd
import numpy as np

# MODEL 03 — ORIGINAL MODEL 02 INFERENCE TEST
# Uses provisional estate mapping for prototype testing only.

df_inference = df_history.copy()

# Match Model 02's expected column name
df_inference["estate_id"] = df_inference["model02_estate_id"]

# Ensure all required historical inputs are available
history_features = [
    "temperature_c_lag1",
    "temperature_c_lag3",
    "temperature_c_lag7",
    "relative_humidity_pct_lag7",
    "rainfall_mm_lag7",
    "soil_moisture_pct_lag3",
    "soil_moisture_pct_prev3d_mean",
    "relative_humidity_pct_prev7d_mean",
    "rainfall_mm_prev7d_mean",
]

df_ready = df_inference.dropna(
    subset=history_features
).copy()

# Select the latest usable record from each estate
latest_rows = (
    df_ready.sort_values("date")
    .groupby("estate_id", as_index=False)
    .tail(1)
    .sort_values("estate_id")
)

supported_diseases = [
    "Blister_Blight",
    "Grey_Blight",
    "Red_Rust",
]

results = []

for _, row in latest_rows.iterrows():
    for disease in supported_diseases:
        # Construct the exact feature schema expected by Model 02
        input_row = {
            feature: row[feature]
            for feature in model02_features
            if feature in row.index
        }
        input_row["disease_class"] = disease
        input_row["estate_id"] = row["estate_id"]
        input_row["month"] = int(row["month"])

        X = pd.DataFrame(
            [[input_row[feature] for feature in model02_features]],
            columns=model02_features,
        )

        probabilities = model02.predict_proba(X)[0]
        class_ids = model02.named_steps["model"].classes_

        # Decode the model's encoded class IDs correctly
        severity_probabilities = {}
        for class_id, probability in zip(class_ids, probabilities):
            severity = severity_encoder.inverse_transform(
                [int(class_id)]
            )[0]
            severity_probabilities[severity] = float(probability)

        predicted_severity = max(
            severity_probabilities,
            key=severity_probabilities.get,
        )

        results.append({
            "date": row["date"].date().isoformat(),
            "estate_name": row["estate_name"],
            "environmental_estate_id": row["estate_id"],
            "disease_class": disease,
            "predicted_severity": predicted_severity,
            **{
                f"prob_{severity}": round(probability, 4)
                for severity, probability
                in severity_probabilities.items()
            },
        })

results_df = pd.DataFrame(results)

print("MODEL 03 — MODEL 02 INFERENCE RESULTS")
print("=" * 65)
print("Prediction rows:", len(results_df))
print("Estate mapping: PROVISIONAL")
print("Predictions are not field-validated.")

display(results_df)


MODEL 03 — MODEL 02 INFERENCE RESULTS
Prediction rows: 15
Estate mapping: PROVISIONAL
Predictions are not field-validated.


,date,estate_name,environmental_estate_id,disease_class,predicted_severity,prob_High,prob_Low,prob_Moderate,prob_Severe
0,2025-12-31,Nuwara Eliya Tea Estate,EST001,Blister_Blight,Severe,0.2908,0.0151,0.0208,0.6734
1,2025-12-31,Nuwara Eliya Tea Estate,EST001,Grey_Blight,Severe,0.2747,0.0140,0.0193,0.6920
2,2025-12-31,Nuwara Eliya Tea Estate,EST001,Red_Rust,Severe,0.2732,0.0144,0.0199,0.6925
3,2025-12-31,Badulla Tea Estate,EST002,Blister_Blight,High,0.7170,0.0139,0.0171,0.2520
4,2025-12-31,Badulla Tea Estate,EST002,Grey_Blight,High,0.6891,0.0132,0.0162,0.2815
5,2025-12-31,Badulla Tea Estate,EST002,Red_Rust,High,0.6872,0.0136,0.0168,0.2824
6,2025-12-31,Kandy Tea Estate,EST003,Blister_Blight,Moderate,0.3654,0.0235,0.4695,0.1417
7,2025-12-31,Kandy Tea Estate,EST003,Grey_Blight,Moderate,0.3611,0.0234,0.4476,0.1679
8,2025-12-31,Kandy Tea Estate,EST003,Red_Rust,Moderate,0.3273,0.0240,0.4813,0.1674
9,2025-12-31,Ratnapura Tea Estate,EST004,Blister_Blight,High,0.6271,0.0189,0.2064,0.1476


In [ ]:

from google.colab import files
from PIL import Image
import numpy as np
import tensorflow as tf

print("Upload one tea-leaf image for Model 01")
uploaded = files.upload()

if not uploaded:
    raise ValueError("No image uploaded.")

image_name = next(iter(uploaded))

# Model 01 expects RGB, 224 × 224 input.
image = Image.open(image_name).convert("RGB").resize((224, 224))
image_array = np.array(image, dtype=np.float32)
image_batch = np.expand_dims(image_array, axis=0)

# Do not divide by 255 externally; use the model's own preprocessing.
probabilities = model01.predict(image_batch, verbose=0)[0]

predicted_index = int(np.argmax(probabilities))
predicted_disease = class_names[predicted_index]
confidence = float(probabilities[predicted_index])

print("\nMODEL 01 — REAL IMAGE PREDICTION")
print("-" * 45)
print("Image:", image_name)
print("Predicted class:", predicted_disease)
print(f"Classification confidence: {confidence:.4f}")

print("\nClass probabilities:")
for label, probability in zip(class_names, probabilities):
    print(f"{label}: {probability:.4f}")


Upload one tea-leaf image for Model 01


Saving 12c8d7eb8a2f.jpg to 12c8d7eb8a2f.jpg

MODEL 01 — REAL IMAGE PREDICTION
---------------------------------------------
Image: 12c8d7eb8a2f.jpg
Predicted class: Blister_Blight
Classification confidence: 0.9894

Class probabilities:
Blister_Blight: 0.9894
Grey_Blight: 0.0106
Red_Rust: 0.0000
Healthy: 0.0000


In [ ]:

import math
import pandas as pd

# MODEL 03 — PDVI FUSION
# Prototype research index, not a validated field-risk probability.

severity_weights = {
    "Low": 0,
    "Moderate": 1,
    "High": 2,
    "Severe": 3,
}

def calculate_expected_severity(row):
    return sum(
        severity_weights[level] * float(row[f"prob_{level}"])
        for level in severity_weights
    ) / 3.0

# Select the severity predictions matching Model 01's disease class
matching = results_df[
    results_df["disease_class"] == predicted_disease
].copy()

if matching.empty:
    raise ValueError(
        f"No Model 02 results found for {predicted_disease}"
    )

matching["expected_severity"] = matching.apply(
    calculate_expected_severity, axis=1
)

matching["pdvi_score"] = matching["expected_severity"].apply(
    lambda s: 100 * math.sqrt(max(0.0, confidence * s))
)

matching["pdvi_score"] = matching["pdvi_score"].clip(0, 100).round(2)

print("MODEL 03 — PROVISIONAL PDVI RESULTS")
print("=" * 65)
print("Model 01 predicted disease:", predicted_disease)
print(f"Model 01 confidence: {confidence:.4f}")
print("Severity weights:", severity_weights)
print("Estate mapping: PROVISIONAL — NOT VERIFIED")
print("PDVI status: Prototype research index")

display(
    matching[
        [
            "date",
            "estate_name",
            "disease_class",
            "predicted_severity",
            "expected_severity",
            "pdvi_score",
        ]
    ].reset_index(drop=True)
)


MODEL 03 — PROVISIONAL PDVI RESULTS
Model 01 predicted disease: Blister_Blight
Model 01 confidence: 0.9894
Severity weights: {'Low': 0, 'Moderate': 1, 'High': 2, 'Severe': 3}
Estate mapping: PROVISIONAL — NOT VERIFIED
PDVI status: Prototype research index


,date,estate_name,disease_class,predicted_severity,expected_severity,pdvi_score
0,2025-12-31,Nuwara Eliya Tea Estate,Blister_Blight,Severe,0.874200,93.00
1,2025-12-31,Badulla Tea Estate,Blister_Blight,High,0.735700,85.32
2,2025-12-31,Kandy Tea Estate,Blister_Blight,Moderate,0.541800,73.22
3,2025-12-31,Ratnapura Tea Estate,Blister_Blight,High,0.634467,79.23
4,2025-12-31,Galle Tea Estate,Blister_Blight,Moderate,0.324167,56.63


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

search_root = "/content/drive/MyDrive"

target_files = {
    "best_tea_disease_efficientnetb0_finetuned.keras",
    "class_names.json",
    "Model02_Final_Model.joblib",
    "Model02_LabelEncoder.joblib",
    "Model02_Feature_List.json",
    "Member01_Model02_Environmental_IoT_Final.csv"
}

found_files = {name: [] for name in target_files}

for root, dirs, files in os.walk(search_root):
    # Avoid scanning unnecessary hidden/system folders.
    dirs[:] = [
        d for d in dirs
        if not d.startswith(".")
    ]

    for filename in files:
        if filename in target_files:
            found_files[filename].append(
                os.path.join(root, filename)
            )

for filename, paths in found_files.items():
    print(f"\n{filename}")
    if paths:
        for path in paths:
            print("  FOUND:", path)
    else:
        print("  NOT FOUND")


Model02_Final_Model.joblib
  FOUND: /content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/Model02_Final/Model02_Final_Model.joblib

Member01_Model02_Environmental_IoT_Final.csv
  FOUND: /content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/Member01_Model02_Environmental_IoT_Final.csv

Model02_LabelEncoder.joblib
  FOUND: /content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/Model02_Final/Model02_LabelEncoder.joblib

best_tea_disease_efficientnetb0_finetuned.keras
  FOUND: /content/drive/MyDrive/Tea_Disease_Research/model_outputs/best_tea_disease_efficientnetb0_finetuned.keras

Model02_Feature_List.json
  FOUND: /content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/Model02_Final/Model02_Feature_List.json

class_names.json
  FOUND: /content/drive/MyDrive/Tea_Disease_Research/model_outputs/class_names.json


In [ ]:

# MODEL 03 — CELL 1: LOAD MODELS AND PREPARE DATA

import os
import json
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

from PIL import Image
from sklearn.pipeline import Pipeline

# --------------------------------------------------
# 1. VERIFIED FILE PATHS
# --------------------------------------------------

MODEL01_PATH = (
    "/content/drive/MyDrive/Tea_Disease_Research/"
    "model_outputs/best_tea_disease_efficientnetb0_finetuned.keras"
)

CLASS_NAMES_PATH = (
    "/content/drive/MyDrive/Tea_Disease_Research/"
    "model_outputs/class_names.json"
)

MODEL02_PATH = (
    "/content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/"
    "Model02_Final/Model02_Final_Model.joblib"
)

ENCODER_PATH = (
    "/content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/"
    "Model02_Final/Model02_LabelEncoder.joblib"
)

FEATURE_LIST_PATH = (
    "/content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/"
    "Model02_Final/Model02_Feature_List.json"
)

ENVIRONMENTAL_CSV_PATH = (
    "/content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/"
    "Member01_Model02_Environmental_IoT_Final.csv"
)

# --------------------------------------------------
# 2. VERIFY FILES
# --------------------------------------------------

required_files = {
    "Model 01": MODEL01_PATH,
    "Class names": CLASS_NAMES_PATH,
    "Model 02": MODEL02_PATH,
    "Label encoder": ENCODER_PATH,
    "Feature list": FEATURE_LIST_PATH,
    "Environmental dataset": ENVIRONMENTAL_CSV_PATH
}

for name, path in required_files.items():
    if not os.path.isfile(path):
        raise FileNotFoundError(f"{name} not found: {path}")
    print(f"FOUND: {name}")

# --------------------------------------------------
# 3. LOAD MODEL 01
# --------------------------------------------------

model01 = tf.keras.models.load_model(
    MODEL01_PATH,
    compile=False
)

with open(CLASS_NAMES_PATH, "r") as f:
    class_names = json.load(f)

print("\nMODEL 01 LOADED")
print("Input shape:", model01.input_shape)
print("Output shape:", model01.output_shape)
print("Class names:", class_names)

# --------------------------------------------------
# 4. LOAD ORIGINAL MODEL 02
# --------------------------------------------------

model02 = joblib.load(MODEL02_PATH)
label_encoder = joblib.load(ENCODER_PATH)

with open(FEATURE_LIST_PATH, "r") as f:
    model02_features = json.load(f)

print("\nMODEL 02 LOADED")
print("Model type:", type(model02).__name__)
print("Features:", len(model02_features))
print("Severity labels:", list(label_encoder.classes_))

# --------------------------------------------------
# 5. LOAD ENVIRONMENTAL DATA
# --------------------------------------------------

env_df = pd.read_csv(ENVIRONMENTAL_CSV_PATH)

env_df["timestamp"] = pd.to_datetime(
    env_df["timestamp"],
    dayfirst=True,
    errors="coerce"
)

if env_df["timestamp"].isna().any():
    raise ValueError(
        "Some environmental timestamps could not be parsed. "
        "Inspect the original CSV before continuing."
    )

env_df = env_df.sort_values(
    ["estate_id", "timestamp"]
).reset_index(drop=True)

print("\nENVIRONMENTAL DATA LOADED")
print("Rows:", len(env_df))
print("Columns:", list(env_df.columns))
print("Estates:", env_df["estate_id"].nunique())
print(
    "Date range:",
    env_df["timestamp"].min().date(),
    "to",
    env_df["timestamp"].max().date()
)

# --------------------------------------------------
# 6. MAP ENVIRONMENTAL ESTATES TO MODEL 02 IDs
# PROVISIONAL MAPPING — VERIFY BEFORE REAL-WORLD USE
# --------------------------------------------------

estate_mapping = {
    "E01": "EST001",
    "E02": "EST002",
    "E03": "EST003",
    "E04": "EST004",
    "E05": "EST005"
}

env_df["model02_estate_id"] = env_df["estate_id"].map(
    estate_mapping
)

if env_df["model02_estate_id"].isna().any():
    unknown_ids = env_df.loc[
        env_df["model02_estate_id"].isna(),
        "estate_id"
    ].unique()
    raise ValueError(f"Unmapped estate IDs: {unknown_ids}")

print("\nCELL 1 COMPLETED SUCCESSFULLY.")


FOUND: Model 01
FOUND: Class names
FOUND: Model 02
FOUND: Label encoder
FOUND: Feature list
FOUND: Environmental dataset

MODEL 01 LOADED
Input shape: (None, 224, 224, 3)
Output shape: (None, 4)
Class names: ['Blister_Blight', 'Grey_Blight', 'Red_Rust', 'Healthy']

MODEL 02 LOADED
Model type: Pipeline
Features: 19
Severity labels: ['High', 'Low', 'Moderate', 'Severe']

ENVIRONMENTAL DATA LOADED
Rows: 9130
Columns: ['timestamp', 'estate_id', 'estate_name', 'temperature_c', 'relative_humidity_pct', 'soil_moisture_pct', 'leaf_wetness', 'rainfall_mm', 'light_intensity_lux', 'wind_speed']
Estates: 5
Date range: 2021-01-01 to 2025-12-31

CELL 1 COMPLETED SUCCESSFULLY.


In [ ]:

# MODEL 03 — CELL 2: HISTORICAL FEATURE ENGINEERING

# Work on a copy; preserve the original environmental dataframe.
df_history = env_df.copy()

df_history = df_history.sort_values(
    ["estate_id", "timestamp"]
).reset_index(drop=True)

# Confirm each estate has no duplicate dates.
duplicates = df_history.duplicated(
    subset=["estate_id", "timestamp"]
).sum()

if duplicates:
    raise ValueError(
        f"Found {duplicates} duplicate estate-date records. "
        "Resolve these before creating historical features."
    )

# Create lag and rolling features within each estate.
grouped = df_history.groupby("estate_id", sort=False)

# Temperature lags
for lag in [1, 3, 7]:
    df_history[f"temperature_c_lag{lag}"] = (
        grouped["temperature_c"].shift(lag)
    )

# Other lag features
df_history["relative_humidity_pct_lag7"] = (
    grouped["relative_humidity_pct"].shift(7)
)

df_history["rainfall_mm_lag7"] = (
    grouped["rainfall_mm"].shift(7)
)

df_history["soil_moisture_pct_lag3"] = (
    grouped["soil_moisture_pct"].shift(3)
)

# Rolling averages: shift by one day first to exclude the current day.
df_history["soil_moisture_pct_prev3d_mean"] = (
    df_history.groupby("estate_id")["soil_moisture_pct"]
    .transform(
        lambda s: s.shift(1).rolling(3, min_periods=3).mean()
    )
)

df_history["relative_humidity_pct_prev7d_mean"] = (
    df_history.groupby("estate_id")["relative_humidity_pct"]
    .transform(
        lambda s: s.shift(1).rolling(7, min_periods=7).mean()
    )
)

df_history["rainfall_mm_prev7d_mean"] = (
    df_history.groupby("estate_id")["rainfall_mm"]
    .transform(
        lambda s: s.shift(1).rolling(7, min_periods=7).mean()
    )
)

# Month feature
df_history["month"] = df_history["timestamp"].dt.month

# Model 02 estate identifier
df_history["model02_estate_id"] = (
    df_history["estate_id"].map(estate_mapping)
)

# Confirm the expected feature columns exist.
missing_features = [
    feature for feature in model02_features
    if feature not in df_history.columns
    and feature not in ["disease_class", "estate_id"]
]

if missing_features:
    raise ValueError(
        f"Missing required Model 02 features: {missing_features}"
    )

# Count records available after historical feature engineering.
history_columns = [
    "temperature_c_lag1",
    "temperature_c_lag3",
    "temperature_c_lag7",
    "relative_humidity_pct_lag7",
    "rainfall_mm_lag7",
    "soil_moisture_pct_lag3",
    "soil_moisture_pct_prev3d_mean",
    "relative_humidity_pct_prev7d_mean",
    "rainfall_mm_prev7d_mean"
]

valid_history = df_history[history_columns].notna().all(axis=1)

print("CELL 2 COMPLETED")
print("Total environmental rows:", len(df_history))
print("Rows with complete historical features:", int(valid_history.sum()))
print("Rows requiring initial historical observations:", int((~valid_history).sum()))

print("\nLatest available record for each estate:")
print(
    df_history.sort_values("timestamp")
    .groupby("estate_id")
    .tail(1)[["estate_id", "estate_name", "timestamp"]]
    .sort_values("estate_id")
    .to_string(index=False)
)


CELL 2 COMPLETED
Total environmental rows: 9130
Rows with complete historical features: 9095
Rows requiring initial historical observations: 35

Latest available record for each estate:
estate_id             estate_name  timestamp
      E01 Nuwara Eliya Tea Estate 2025-12-31
      E02      Badulla Tea Estate 2025-12-31
      E03        Kandy Tea Estate 2025-12-31
      E04    Ratnapura Tea Estate 2025-12-31
      E05        Galle Tea Estate 2025-12-31


In [ ]:

# MODEL 03 — CELL 3: INTEGRATED PREDICTION FUNCTION

from datetime import datetime
from PIL import Image
import numpy as np
import pandas as pd
import json
import os

# --------------------------------------------------
# CONFIGURATION
# --------------------------------------------------

SEVERITY_WEIGHTS = {
    "Low": 0,
    "Moderate": 1,
    "High": 2,
    "Severe": 3
}

# Provisional mapping; verify against the original Model 02 training data.
ESTATE_MAPPING = estate_mapping.copy()


def predict_model03(image_path, estate_id, prediction_date):
    """
    Integrate Model 01 leaf classification with Model 02
    conditional severity prediction and provisional PDVI.

    prediction_date format: YYYY-MM-DD
    estate_id: E01, E02, E03, E04, or E05
    """

    # 1. Validate inputs
    if not os.path.isfile(image_path):
        raise FileNotFoundError(f"Image not found: {image_path}")

    if estate_id not in ESTATE_MAPPING:
        raise ValueError(
            f"Unknown estate_id: {estate_id}. "
            f"Choose from {list(ESTATE_MAPPING.keys())}"
        )

    prediction_date = pd.to_datetime(
        prediction_date, errors="raise"
    ).normalize()

    model02_estate_id = ESTATE_MAPPING[estate_id]

    # 2. Find the environmental record for the requested date.
    estate_history = df_history[
        df_history["estate_id"] == estate_id
    ].sort_values("timestamp")

    eligible_history = estate_history[
        (estate_history["timestamp"] <= prediction_date)
        & estate_history[
            [
                "temperature_c_lag1",
                "temperature_c_lag3",
                "temperature_c_lag7",
                "relative_humidity_pct_lag7",
                "rainfall_mm_lag7",
                "soil_moisture_pct_lag3",
                "soil_moisture_pct_prev3d_mean",
                "relative_humidity_pct_prev7d_mean",
                "rainfall_mm_prev7d_mean"
            ]
        ].notna().all(axis=1)
    ]

    if eligible_history.empty:
        raise ValueError(
            "No usable environmental record exists on or before "
            f"{prediction_date.date()} for {estate_id}. "
            "Check the date and historical data."
        )

    # Use the latest available record on or before the requested date.
    # Report the actual data date to avoid hiding stale environmental data.
    env_row = eligible_history.iloc[-1]
    environment_date = env_row["timestamp"].normalize()

    # 3. Model 01 — classify the uploaded leaf image.
    image = Image.open(image_path).convert("RGB").resize((224, 224))
    image_array = np.asarray(image, dtype=np.float32)
    image_array = np.expand_dims(image_array, axis=0)

    model01_probabilities = model01.predict(
        image_array, verbose=0
    )[0]

    predicted_index = int(np.argmax(model01_probabilities))
    predicted_disease = class_names[predicted_index]
    disease_confidence = float(
        model01_probabilities[predicted_index]
    )

    model01_results = {
        name: float(probability)
        for name, probability in zip(
            class_names, model01_probabilities
        )
    }

    # 4. Healthy leaves do not receive a disease-specific PDVI here.
    # This does not mean the whole estate has zero disease risk.
    if predicted_disease == "Healthy":
        return {
            "prediction_date_requested": str(prediction_date.date()),
            "environment_date_used": str(environment_date.date()),
            "estate_id": estate_id,
            "model02_estate_id": model02_estate_id,
            "predicted_disease": predicted_disease,
            "disease_confidence": disease_confidence,
            "model01_probabilities": model01_results,
            "severity_prediction": None,
            "severity_probabilities": None,
            "expected_severity_0_to_3": None,
            "pdvi_0_to_100": None,
            "interpretation": (
                "Model 01 classified this image as Healthy. "
                "A disease-specific PDVI was not calculated. "
                "This does not establish that the estate is disease-free."
            )
        }

    # 5. Prepare the exact features required by Model 02.
    row = {}

    for feature in model02_features:
        if feature == "disease_class":
            row[feature] = predicted_disease
        elif feature == "estate_id":
            row[feature] = model02_estate_id
        elif feature == "month":
            row[feature] = int(env_row["timestamp"].month)
        else:
            if feature not in env_row.index:
                raise KeyError(
                    f"Required feature '{feature}' is missing "
                    "from the environmental history."
                )
            row[feature] = env_row[feature]

    model02_input = pd.DataFrame(
        [row], columns=model02_features
    )

    # 6. Model 02 — predict severity probabilities.
    severity_probabilities_raw = model02.predict_proba(
        model02_input
    )[0]

    model02_classes = model02.classes_

    # Decode each encoded class ID into its severity label.
    severity_probabilities = {}

    for encoded_class, probability in zip(
        model02_classes, severity_probabilities_raw
    ):
        decoded_label = label_encoder.inverse_transform(
            [int(encoded_class)]
        )[0]

        severity_probabilities[str(decoded_label)] = float(
            probability
        )

    # Ensure all expected severity classes are present.
    for severity in SEVERITY_WEIGHTS:
        severity_probabilities.setdefault(severity, 0.0)

    # 7. Calculate expected severity on a 0–3 scale.
    expected_severity = sum(
        SEVERITY_WEIGHTS[label] * probability
        for label, probability in severity_probabilities.items()
        if label in SEVERITY_WEIGHTS
    ) / 3.0

    expected_severity = float(
        np.clip(expected_severity, 0.0, 1.0)
    )

    # 8. Provisional PDVI formula.
    pdvi = 100.0 * np.sqrt(
        np.clip(disease_confidence * expected_severity, 0.0, 1.0)
    )

    # The severity label is the highest-probability class.
    predicted_severity = max(
        severity_probabilities,
        key=severity_probabilities.get
    )

    return {
        "prediction_date_requested": str(prediction_date.date()),
        "environment_date_used": str(environment_date.date()),
        "estate_id": estate_id,
        "model02_estate_id": model02_estate_id,
        "predicted_disease": predicted_disease,
        "disease_confidence": disease_confidence,
        "model01_probabilities": model01_results,
        "severity_prediction": predicted_severity,
        "severity_probabilities": severity_probabilities,
        "expected_severity_0_to_3": round(
            expected_severity * 3.0, 4
        ),
        "pdvi_0_to_100": round(float(pdvi), 2),
        "pdvi_formula": "100 * sqrt(disease_confidence * expected_severity_0_to_1)",
        "interpretation": (
            "Provisional image-conditioned severity index, "
            "not a validated probability of disease occurrence "
            "or a calibrated plantation-wide risk score."
        )
    }


print("CELL 3 COMPLETED: predict_model03() is defined.")


CELL 3 COMPLETED: predict_model03() is defined.


In [ ]:

import os

# Search common locations first
search_roots = [
    "/content",
    "/content/drive/MyDrive"
]

target = "12c8d7eb8a2f.jpg"
matches = []

for base in search_roots:
    if os.path.exists(base):
        for root, dirs, files in os.walk(base):
            dirs[:] = [
                d for d in dirs
                if d not in {"__pycache__", ".ipynb_checkpoints"}
            ]
            if target in files:
                matches.append(os.path.join(root, target))

if matches:
    print("Image found:")
    for path in matches:
        print(path)
else:
    print("Image not found. Upload the tea-leaf test image below.")


Image found:
/content/drive/MyDrive/Tea_Disease_Research/Model03_Outputs/test_images/12c8d7eb8a2f.jpg
/content/drive/MyDrive/Tea_Disease_Research/Model03_Backup_20261020_012757/test_images/12c8d7eb8a2f.jpg
/content/drive/MyDrive/Tea_Disease_Research/Model03_Outputs/test_images/12c8d7eb8a2f.jpg
/content/drive/MyDrive/Tea_Disease_Research/Model03_Backup_20261020_012757/test_images/12c8d7eb8a2f.jpg


In [ ]:

# MODEL 03 — CELL 4: FIRST INTEGRATION TEST

image_path = (
    "/content/drive/MyDrive/Tea_Disease_Research/"
    "Model03_Outputs/test_images/12c8d7eb8a2f.jpg"
)

result = predict_model03(
    image_path=image_path,
    estate_id="E01",
    prediction_date="2025-12-31"
)

print(json.dumps(result, indent=2))


{
  "prediction_date_requested": "2025-12-31",
  "environment_date_used": "2025-12-31",
  "estate_id": "E01",
  "model02_estate_id": "EST001",
  "predicted_disease": "Blister_Blight",
  "disease_confidence": 0.9894363880157471,
  "model01_probabilities": {
    "Blister_Blight": 0.9894363880157471,
    "Grey_Blight": 0.010562832467257977,
    "Red_Rust": 9.484795171488258e-10,
    "Healthy": 7.740752039353538e-07
  },
  "severity_prediction": "Severe",
  "severity_probabilities": {
    "High": 0.2907885700066613,
    "Low": 0.015068726917811616,
    "Moderate": 0.020765067010258726,
    "Severe": 0.6733776360652685
  },
  "expected_severity_0_to_3": 2.6225,
  "pdvi_0_to_100": 93.0,
  "pdvi_formula": "100 * sqrt(disease_confidence * expected_severity_0_to_1)",
  "interpretation": "Provisional image-conditioned severity index, not a validated probability of disease occurrence or a calibrated plantation-wide risk score."
}


In [ ]:

# MODEL 03 — CELL 5: FIVE-ESTATE INTEGRATION TEST

import pandas as pd
from IPython.display import display

test_image = (
    "/content/drive/MyDrive/Tea_Disease_Research/"
    "Model03_Outputs/test_images/12c8d7eb8a2f.jpg"
)

test_date = "2025-12-31"

estate_names = (
    env_df[["estate_id", "estate_name"]]
    .drop_duplicates()
    .sort_values("estate_id")
)

results = []

for _, estate in estate_names.iterrows():
    try:
        output = predict_model03(
            image_path=test_image,
            estate_id=estate["estate_id"],
            prediction_date=test_date
        )

        results.append({
            "estate_id": output["estate_id"],
            "estate_name": estate["estate_name"],
            "environment_date": output["environment_date_used"],
            "predicted_disease": output["predicted_disease"],
            "disease_confidence_pct": round(
                output["disease_confidence"] * 100, 2
            ),
            "predicted_severity": output["severity_prediction"],
            "expected_severity_0_to_3": output[
                "expected_severity_0_to_3"
            ],
            "pdvi": output["pdvi_0_to_100"],
            "status": "SUCCESS"
        })

    except Exception as e:
        results.append({
            "estate_id": estate["estate_id"],
            "estate_name": estate["estate_name"],
            "status": f"FAILED: {e}"
        })

results_df = pd.DataFrame(results)

display(results_df)

print("\nSuccessful predictions:",
      (results_df["status"] == "SUCCESS").sum(),
      "out of", len(results_df))


,estate_id,estate_name,environment_date,predicted_disease,disease_confidence_pct,predicted_severity,expected_severity_0_to_3,pdvi,status
0,E01,Nuwara Eliya Tea Estate,2025-12-31,Blister_Blight,98.94,Severe,2.6225,93.00,SUCCESS
1,E02,Badulla Tea Estate,2025-12-31,Blister_Blight,98.94,High,2.2070,85.32,SUCCESS
2,E03,Kandy Tea Estate,2025-12-31,Blister_Blight,98.94,Moderate,1.6253,73.22,SUCCESS
3,E04,Ratnapura Tea Estate,2025-12-31,Blister_Blight,98.94,High,1.9035,79.23,SUCCESS
4,E05,Galle Tea Estate,2025-12-31,Blister_Blight,98.94,Moderate,0.9725,56.63,SUCCESS



Successful predictions: 5 out of 5


In [ ]:

# MODEL 03 — CELL 6: VALIDATE AND SAVE TEST RESULTS

import os
import json
from datetime import datetime
import pandas as pd
import numpy as np

# 1. Validate the five-estate results
assert len(results_df) == 5, "Expected five estate results."
assert (results_df["status"] == "SUCCESS").all(), (
    "At least one estate prediction failed."
)

assert results_df["pdvi"].notna().all(), (
    "Some PDVI values are missing; inspect results before saving."
)

assert results_df["pdvi"].between(0, 100).all(), (
    "PDVI values must be between 0 and 100."
)

assert (results_df["environment_date"] == test_date).all(), (
    "At least one estate used a different environmental date."
)

assert results_df["predicted_disease"].nunique() == 1, (
    "The same image produced inconsistent disease classifications."
)

# 2. Create a NEW timestamped output directory
base_output_dir = (
    "/content/drive/MyDrive/Tea_Disease_Research/"
    "Model03_Outputs"
)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = os.path.join(
    base_output_dir, f"Integration_Test_{timestamp}"
)

os.makedirs(output_dir, exist_ok=False)

# 3. Save the summary CSV
csv_path = os.path.join(
    output_dir, "Model03_Five_Estate_Integration_Test.csv"
)

results_df.to_csv(csv_path, index=False)

# 4. Save the detailed results as JSON
json_path = os.path.join(
    output_dir, "Model03_Five_Estate_Integration_Test.json"
)

with open(json_path, "w") as f:
    json.dump(
        {
            "test_type": "Five-estate integration test",
            "prediction_date": test_date,
            "image_path": test_image,
            "number_of_estates": len(results_df),
            "successful_predictions": int(
                (results_df["status"] == "SUCCESS").sum()
            ),
            "pdvi_formula": (
                "100 * sqrt(disease_confidence * "
                "expected_severity_0_to_1)"
            ),
            "validation_status": (
                "Integration test passed; scientific validation pending."
            ),
            "results": results_df.to_dict(orient="records")
        },
        f,
        indent=2
    )

print("VALIDATION PASSED")
print("Output directory:", output_dir)
print("CSV saved:", csv_path)
print("JSON saved:", json_path)
print("Existing Model03 outputs were not overwritten.")


VALIDATION PASSED
Output directory: /content/drive/MyDrive/Tea_Disease_Research/Model03_Outputs/Integration_Test_20261020_035432
CSV saved: /content/drive/MyDrive/Tea_Disease_Research/Model03_Outputs/Integration_Test_20261020_035432/Model03_Five_Estate_Integration_Test.csv
JSON saved: /content/drive/MyDrive/Tea_Disease_Research/Model03_Outputs/Integration_Test_20261020_035432/Model03_Five_Estate_Integration_Test.json
Existing Model03 outputs were not overwritten.


In [ ]:

# MODEL 03 — STEP 7A: AUDIT MODEL 02 CONFIGURATION
# Read-only inspection. Does not retrain or modify either model.

import json
import numpy as np
import pandas as pd

print("=" * 65)
print("MODEL 02 CONFIGURATION AUDIT")
print("=" * 65)

# 1. Confirm the exact input feature order.
print("\n1. REQUIRED INPUT FEATURES")
for i, feature in enumerate(model02_features, start=1):
    print(f"{i:02d}. {feature}")

# 2. Inspect the loaded pipeline.
print("\n2. PIPELINE COMPONENTS")
print("Pipeline type:", type(model02).__name__)

if hasattr(model02, "named_steps"):
    for step_name, step_object in model02.named_steps.items():
        print(f"{step_name}: {type(step_object).__name__}")
else:
    print("WARNING: No named_steps attribute found.")

# 3. Inspect categorical preprocessing and learned categories.
print("\n3. PREPROCESSING AND LEARNED CATEGORIES")

if hasattr(model02, "named_steps"):
    preprocessor = model02.named_steps.get("preprocessor")

    if preprocessor is not None and hasattr(preprocessor, "transformers_"):
        for name, transformer, columns in preprocessor.transformers_:
            print(f"\nTransformer: {name}")
            print("Columns:", columns)

            if hasattr(transformer, "categories_"):
                for col, categories in zip(
                    columns, transformer.categories_
                ):
                    print(f"  {col}: {list(categories)}")

            elif hasattr(transformer, "named_steps"):
                for sub_name, sub_transformer in (
                    transformer.named_steps.items()
                ):
                    print(f"  Sub-step: {sub_name}")
                    print(
                        "  Type:",
                        type(sub_transformer).__name__
                    )

                    if hasattr(sub_transformer, "categories_"):
                        for col, categories in zip(
                            columns, sub_transformer.categories_
                        ):
                            print(
                                f"  {col}: {list(categories)}"
                            )
    else:
        print(
            "Preprocessor not found or fitted transformer details "
            "are unavailable."
        )

# 4. Inspect the label encoder.
print("\n4. SEVERITY LABEL ENCODER")
for encoded_id, label in enumerate(label_encoder.classes_):
    print(f"{encoded_id} -> {label}")

# 5. Inspect environmental estate identifiers.
print("\n5. ESTATE IDENTIFIERS IN ENVIRONMENTAL DATA")
print(
    env_df[["estate_id", "estate_name"]]
    .drop_duplicates()
    .sort_values("estate_id")
    .to_string(index=False)
)

print("\nPROVISIONAL MAPPING CURRENTLY USED")
print(json.dumps(ESTATE_MAPPING, indent=2))

print("\nAUDIT CELL COMPLETED — NO FILES MODIFIED.")


MODEL 02 CONFIGURATION AUDIT

1. REQUIRED INPUT FEATURES
01. temperature_c
02. relative_humidity_pct
03. soil_moisture_pct
04. leaf_wetness
05. rainfall_mm
06. light_intensity_lux
07. wind_speed
08. temperature_c_lag1
09. temperature_c_lag3
10. temperature_c_lag7
11. relative_humidity_pct_lag7
12. rainfall_mm_lag7
13. soil_moisture_pct_lag3
14. soil_moisture_pct_prev3d_mean
15. relative_humidity_pct_prev7d_mean
16. rainfall_mm_prev7d_mean
17. month
18. disease_class
19. estate_id

2. PIPELINE COMPONENTS
Pipeline type: Pipeline
preprocessor: ColumnTransformer
model: HistGradientBoostingClassifier

3. PREPROCESSING AND LEARNED CATEGORIES

Transformer: numeric
Columns: ['temperature_c', 'relative_humidity_pct', 'soil_moisture_pct', 'leaf_wetness', 'rainfall_mm', 'light_intensity_lux', 'wind_speed', 'temperature_c_lag1', 'temperature_c_lag3', 'temperature_c_lag7', 'relative_humidity_pct_lag7', 'rainfall_mm_lag7', 'soil_moisture_pct_lag3', 'soil_moisture_pct_prev3d_mean', 'relative_humidity

In [ ]:

# MODEL 03 — STEP 7B: SEARCH FOR TRAINING-DATA EVIDENCE

import os

drive_root = "/content/drive/MyDrive"

keywords = [
    "model02",
    "severity",
    "training",
    "train",
    "dataset",
    "estate",
    "mapping",
    "metadata"
]

extensions = (".csv", ".xlsx", ".json", ".txt", ".joblib", ".pkl")

matches = []

for root, dirs, files in os.walk(drive_root):
    dirs[:] = [
        d for d in dirs
        if not d.startswith(".")
        and "Model03_Backup_" not in d
    ]

    for filename in files:
        lower_name = filename.lower()

        if (
            lower_name.endswith(extensions)
            and any(word in lower_name for word in keywords)
        ):
            matches.append(os.path.join(root, filename))

print("Potentially relevant files:", len(matches))

for path in sorted(matches)[:150]:
    print(path)

if len(matches) > 150:
    print("Showing first 150 matches only.")


Potentially relevant files: 19
/content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/Member01_Model02_Disease_Occurrence_Severity_Final.csv
/content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/Member01_Model02_Environmental_IoT_Final.csv
/content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/Model02_Aligned_V2/Model02_Aligned_V2_Evaluation_Report.json
/content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/Model02_Aligned_V2/Model02_Aligned_V2_Feature_List.json
/content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/Model02_Aligned_V2/Model02_Aligned_V2_LabelEncoder.joblib
/content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/Model02_Aligned_V2/Model02_Aligned_V2_Model.joblib
/content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/Model02_Aligned_V2/Model02_Aligned_V2_Training_Data.csv
/content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/Model02_Final/Model02_Feature_List.json
/content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/Model02

In [ ]:

import os
import json
import pandas as pd

base = "/content/drive/MyDrive/Disease_Severity_And_Weather_Dataset"

files_to_inspect = {
    "severity_training": os.path.join(
        base, "Severity_Training_Data.csv"
    ),
    "occurrence_severity": os.path.join(
        base, "Member01_Model02_Disease_Occurrence_Severity_Final.csv"
    ),
    "model02_evaluation": os.path.join(
        base, "Model02_Final", "Model02_Final_Evaluation_Report.json"
    )
}

# 1. Inspect training CSV schemas and estate information
for name in ["severity_training", "occurrence_severity"]:
    path = files_to_inspect[name]
    print("\n" + "=" * 60)
    print(name, "—", path)

    if not os.path.isfile(path):
        print("FILE NOT FOUND")
        continue

    df = pd.read_csv(path, nrows=100000)
    print("Rows inspected:", len(df))
    print("Columns:", df.columns.tolist())

    for col in df.columns:
        if any(word in col.lower() for word in [
            "estate", "district", "location", "severity", "disease"
        ]):
            print(f"\nValues for {col}:")
            print(df[col].value_counts(dropna=False).head(15).to_string())

# 2. Inspect the original Model 02 evaluation report
path = files_to_inspect["model02_evaluation"]

print("\n" + "=" * 60)
print("MODEL 02 EVALUATION REPORT")

if os.path.isfile(path):
    with open(path, "r") as f:
        report = json.load(f)
    print(json.dumps(report, indent=2)[:12000])
else:
    print("Evaluation report not found.")



severity_training — /content/drive/MyDrive/Disease_Severity_And_Weather_Dataset/Severity_Training_Data.csv
Rows inspected: 4000
Columns: ['record_id', 'date', 'estate_id', 'estate_name', 'disease_class', 'severity_level', 'temperature_c', 'relative_humidity_pct', 'soil_moisture_pct', 'leaf_wetness', 'rainfall_mm', 'light_intensity_lux', 'wind_speed', 'temperature_c_lag1', 'temperature_c_lag3', 'temperature_c_lag7', 'relative_humidity_pct_lag7', 'rainfall_mm_lag7', 'soil_moisture_pct_lag3', 'soil_moisture_pct_prev3d_mean', 'relative_humidity_pct_prev7d_mean', 'rainfall_mm_prev7d_mean', 'month', 'affected_plants_count', 'total_plants_observed', 'disease_incidence_pct', 'affected_area_pct']

Values for estate_id:
estate_id
EST003    820
EST002    808
EST001    803
EST005    785
EST004    784

Values for estate_name:
estate_name
Matara         820
Passara        808
Talawakelle    803
Kandy          785
Ratnapura      784

Values for disease_class:
disease_class
Blister_Blight    1686
Gre

In [ ]:

# MODEL 03 — ESTATE MAPPING DOCUMENTATION
# Creates a new file; does not modify existing outputs.

import os
import json
from datetime import datetime

audit = {
    "project": "SmartTea-LK Model 03",
    "purpose": "Prototype estate-ID mapping documentation",
    "mapping_status": "Provisional; geographical equivalence unverified",
    "mapping_used": {
        "E01": "EST001",
        "E02": "EST002",
        "E03": "EST003",
        "E04": "EST004",
        "E05": "EST005"
    },
    "environmental_estates": {
        "E01": "Nuwara Eliya Tea Estate",
        "E02": "Badulla Tea Estate",
        "E03": "Kandy Tea Estate",
        "E04": "Ratnapura Tea Estate",
        "E05": "Galle Tea Estate"
    },
    "model02_training_estate_names": {
        "EST001": "Talawakelle",
        "EST002": "Passara",
        "EST003": "Matara",
        "EST004": "Ratnapura",
        "EST005": "Kandy"
    },
    "model02_dataset_type": "Synthetic",
    "model02_real_world_validation": "Pending",
    "pdvi_status": "Provisional; not scientifically calibrated",
    "recorded_at": datetime.now().isoformat(timespec="seconds")
}

audit_dir = (
    "/content/drive/MyDrive/Tea_Disease_Research/"
    "Model03_Outputs"
)

os.makedirs(audit_dir, exist_ok=True)

audit_path = os.path.join(
    audit_dir,
    f"Model03_Estate_Mapping_Audit_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
)

with open(audit_path, "x") as f:
    json.dump(audit, f, indent=2)

print("Estate mapping audit saved:")
print(audit_path)
print("\nExisting models and output files were not modified.")


Estate mapping audit saved:
/content/drive/MyDrive/Tea_Disease_Research/Model03_Outputs/Model03_Estate_Mapping_Audit_20261020_040212.json

Existing models and output files were not modified.


In [ ]:

# MODEL 03 — STEP 10: FINAL EVALUATION PACKAGE
# Uses existing predictions. Does not retrain models or overwrite outputs.

import os
import json
import pandas as pd
from datetime import datetime

base_dir = (
    "/content/drive/MyDrive/Tea_Disease_Research/"
    "Model03_Outputs"
)

test_dir = os.path.join(
    base_dir,
    "Integration_Test_20261020_035432"
)

csv_input = os.path.join(
    test_dir,
    "Model03_Five_Estate_Integration_Test.csv"
)

json_input = os.path.join(
    test_dir,
    "Model03_Five_Estate_Integration_Test.json"
)

mapping_audit_path = (
    "/content/drive/MyDrive/Tea_Disease_Research/"
    "Model03_Outputs/"
    "Model03_Estate_Mapping_Audit_20261020_040212.json"
)

# 1. Verify all inputs exist
required = [csv_input, json_input, mapping_audit_path]

for path in required:
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Required file not found: {path}")
    print("FOUND:", path)

# 2. Load saved results and audit
results = pd.read_csv(csv_input)

with open(json_input, "r") as f:
    test_metadata = json.load(f)

with open(mapping_audit_path, "r") as f:
    mapping_audit = json.load(f)

# 3. Validate integration results
if len(results) != 5:
    raise ValueError(f"Expected 5 estate rows, found {len(results)}")

if not (results["status"] == "SUCCESS").all():
    raise ValueError("One or more estate predictions did not succeed.")

if not results["pdvi"].between(0, 100).all():
    raise ValueError("A PDVI score falls outside the range 0–100.")

if results["environment_date"].nunique() != 1:
    raise ValueError("The test results use inconsistent environment dates.")

if results["predicted_disease"].nunique() != 1:
    raise ValueError("The same image produced different disease classes.")

# 4. Produce a concise evaluation summary
summary = {
    "project": "SmartTea-LK",
    "component": "Model 03 — Integrated Prototype",
    "test_type": "Five-estate integration test",
    "test_date": test_metadata.get("prediction_date"),
    "estates_tested": int(len(results)),
    "successful_predictions": int(
        (results["status"] == "SUCCESS").sum()
    ),
    "integration_test_passed": True,
    "predicted_disease_for_test_image": str(
        results["predicted_disease"].iloc[0]
    ),
    "image_confidence_pct": float(
        results["disease_confidence_pct"].iloc[0]
    ),
    "pdvi_scores": {
        str(row["estate_id"]): float(row["pdvi"])
        for _, row in results.iterrows()
    },
    "minimum_pdvi": float(results["pdvi"].min()),
    "maximum_pdvi": float(results["pdvi"].max()),
    "mean_pdvi": float(results["pdvi"].mean()),
    "estate_mapping_status": mapping_audit["mapping_status"],
    "model02_dataset_type": "Synthetic",
    "model02_real_world_validation": "Pending",
    "pdvi_scientific_calibration": "Pending",
    "interpretation": (
        "Successful software integration test only. These results "
        "do not establish real-world plantation disease risk, "
        "because the same leaf image was used across all estates, "
        "the estate mapping is provisional, and Model 02 real-world "
        "validation and PDVI calibration remain pending."
    )
}

# 5. Create a new, unique output directory
run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
package_dir = os.path.join(
    base_dir,
    f"Final_Evaluation_Package_{run_stamp}"
)

os.makedirs(package_dir, exist_ok=False)

# 6. Save independent copies of the summary and detailed results
summary_json = os.path.join(
    package_dir, "Model03_Final_Evaluation_Summary.json"
)

summary_csv = os.path.join(
    package_dir, "Model03_Final_Evaluation_Summary.csv"
)

detailed_csv = os.path.join(
    package_dir, "Model03_Five_Estate_Results.csv"
)

with open(summary_json, "x") as f:
    json.dump(summary, f, indent=2)

pd.DataFrame([summary]).to_csv(summary_csv, index=False)
results.to_csv(detailed_csv, index=False)

print("\nEVALUATION PACKAGE CREATED")
print("Folder:", package_dir)
print("Summary JSON:", summary_json)
print("Summary CSV:", summary_csv)
print("Detailed results CSV:", detailed_csv)
print("\nIntegration test: PASSED")
print("Scientific validation: PENDING")
print("Original models and existing results were not modified.")


FOUND: /content/drive/MyDrive/Tea_Disease_Research/Model03_Outputs/Integration_Test_20261020_035432/Model03_Five_Estate_Integration_Test.csv
FOUND: /content/drive/MyDrive/Tea_Disease_Research/Model03_Outputs/Integration_Test_20261020_035432/Model03_Five_Estate_Integration_Test.json
FOUND: /content/drive/MyDrive/Tea_Disease_Research/Model03_Outputs/Model03_Estate_Mapping_Audit_20261020_040212.json

EVALUATION PACKAGE CREATED
Folder: /content/drive/MyDrive/Tea_Disease_Research/Model03_Outputs/Final_Evaluation_Package_20261020_040237
Summary JSON: /content/drive/MyDrive/Tea_Disease_Research/Model03_Outputs/Final_Evaluation_Package_20261020_040237/Model03_Final_Evaluation_Summary.json
Summary CSV: /content/drive/MyDrive/Tea_Disease_Research/Model03_Outputs/Final_Evaluation_Package_20261020_040237/Model03_Final_Evaluation_Summary.csv
Detailed results CSV: /content/drive/MyDrive/Tea_Disease_Research/Model03_Outputs/Final_Evaluation_Package_20261020_040237/Model03_Five_Estate_Results.csv

Int

In [ ]:

import os

ROOT = "/content/drive/MyDrive/Tea_Disease_Research"

print("MODEL 03 FILE INVENTORY")
print("=" * 70)

for folder in ["Model03_Outputs", "Model03_Backup_20261020_012757"]:
    path = os.path.join(ROOT, folder)

    if not os.path.isdir(path):
        print(f"\nFolder not found: {path}")
        continue

    print(f"\n📁 {folder}")

    for current, dirs, files in os.walk(path):
        dirs.sort()
        files.sort()

        relative = os.path.relpath(current, path)
        if relative != ".":
            print(f"\n  [{relative}]")

        for filename in files:
            full_path = os.path.join(current, filename)
            size_mb = os.path.getsize(full_path) / (1024 * 1024)
            print(f"    {filename}  ({size_mb:.2f} MB)")

print("\nInventory complete. No files were changed.")


MODEL 03 FILE INVENTORY

📁 Model03_Outputs
    Model03_Estate_Mapping_Audit_20261020_040212.json  (0.00 MB)
    Model03_Final_Integrated_Report.json  (0.00 MB)
    Model03_Final_Integrated_Summary.csv  (0.00 MB)
    Model03_Latest_Prediction.json  (0.00 MB)
    Model03_Model02_Feature_List.json  (0.00 MB)
    Model03_PDVI_Config.json  (0.00 MB)
    Model03_PDVI_Config_v2.json  (0.00 MB)
    Model03_Second_Test_Prediction.json  (0.00 MB)
    model03_integrated_result.json  (0.00 MB)
    model03_predictor.py  (0.01 MB)
    model03_predictor_v2.py  (0.01 MB)

  [Final_Evaluation_Package_20261020_040237]
    Model03_Final_Evaluation_Summary.csv  (0.00 MB)
    Model03_Final_Evaluation_Summary.json  (0.00 MB)
    Model03_Five_Estate_Results.csv  (0.00 MB)

  [Integration_Test_20261020_035432]
    Model03_Five_Estate_Integration_Test.csv  (0.00 MB)
    Model03_Five_Estate_Integration_Test.json  (0.00 MB)

  [__pycache__]
    model03_predictor.cpython-313.pyc  (0.01 MB)
    model03_predictor_v

In [ ]:

import os
import shutil
from datetime import datetime

ROOT = "/content/drive/MyDrive/Tea_Disease_Research"
OUT = os.path.join(ROOT, "Model03_Outputs")

archive_name = "Model03_Archive_" + datetime.now().strftime("%Y%m%d_%H%M%S")
ARCHIVE = os.path.join(OUT, archive_name)
os.makedirs(ARCHIVE, exist_ok=False)

files_to_archive = [
    "Model03_Final_Integrated_Report.json",
    "Model03_Final_Integrated_Summary.csv",
    "Model03_Latest_Prediction.json",
    "Model03_PDVI_Config.json",
    "Model03_PDVI_Config_v2.json",
    "Model03_Second_Test_Prediction.json",
    "model03_integrated_result.json",
    "model03_predictor.py",
    "model03_predictor_v2.py",
]

copied = []

for filename in files_to_archive:
    source = os.path.join(OUT, filename)

    if os.path.isfile(source):
        destination = os.path.join(ARCHIVE, filename)
        shutil.copy2(source, destination)

        # Verify copied file matches original
        if os.path.getsize(source) != os.path.getsize(destination):
            raise IOError(f"Copy verification failed: {filename}")

        copied.append(filename)
        print("ARCHIVED COPY:", filename)

print("\nArchive created:", ARCHIVE)
print("Files copied:", len(copied))
print("Original files were NOT deleted or modified.")


ARCHIVED COPY: Model03_Final_Integrated_Report.json
ARCHIVED COPY: Model03_Final_Integrated_Summary.csv
ARCHIVED COPY: Model03_Latest_Prediction.json
ARCHIVED COPY: Model03_PDVI_Config.json
ARCHIVED COPY: Model03_PDVI_Config_v2.json
ARCHIVED COPY: Model03_Second_Test_Prediction.json
ARCHIVED COPY: model03_integrated_result.json
ARCHIVED COPY: model03_predictor.py
ARCHIVED COPY: model03_predictor_v2.py

Archive created: /content/drive/MyDrive/Tea_Disease_Research/Model03_Outputs/Model03_Archive_20261020_040550
Files copied: 9
Original files were NOT deleted or modified.


In [ ]:

import os
import shutil

ROOT = "/content/drive/MyDrive/Tea_Disease_Research"
OUT = os.path.join(ROOT, "Model03_Outputs")
OLD = os.path.join(OUT, "Model03_Old")

os.makedirs(OLD, exist_ok=True)

# Older files to move out of the main folder
old_files = [
    "Model03_Final_Integrated_Report.json",
    "Model03_Final_Integrated_Summary.csv",
    "Model03_Latest_Prediction.json",
    "Model03_PDVI_Config.json",
    "Model03_PDVI_Config_v2.json",
    "Model03_Second_Test_Prediction.json",
    "model03_integrated_result.json",
    "model03_predictor.py",
    "model03_predictor_v2.py",
]

moved = []

for name in old_files:
    source = os.path.join(OUT, name)
    destination = os.path.join(OLD, name)

    if os.path.isfile(source):
        if os.path.exists(destination):
            print("SKIPPED — destination already exists:", name)
            continue

        shutil.move(source, destination)
        moved.append(name)
        print("MOVED:", name)

# Move the previous archive into Model03_Old
previous_archive = os.path.join(
    OUT, "Model03_Archive_20261020_040550"
)
archive_destination = os.path.join(
    OLD, "Model03_Archive_20261020_040550"
)

if os.path.isdir(previous_archive):
    if not os.path.exists(archive_destination):
        shutil.move(previous_archive, archive_destination)
        print("MOVED previous archive into Model03_Old")
    else:
        print("Previous archive already exists inside Model03_Old; left original untouched.")

# Remove only disposable Python cache
cache_dir = os.path.join(OUT, "__pycache__")

if os.path.isdir(cache_dir):
    shutil.rmtree(cache_dir)
    print("REMOVED disposable __pycache__")

# Verify the final layout
print("\n" + "=" * 55)
print("CLEANUP COMPLETE — VERIFYING MAIN FOLDER")
print("=" * 55)

for item in sorted(os.listdir(OUT)):
    print(("📁 " if os.path.isdir(os.path.join(OUT, item)) else "📄 ") + item)

print("\nFiles moved:", len(moved))
print("Old files are preserved in:", OLD)
print("Model 01/02 folders and the separate Model03 backup were not touched.")


MOVED: Model03_Final_Integrated_Report.json
MOVED: Model03_Final_Integrated_Summary.csv
MOVED: Model03_Latest_Prediction.json
MOVED: Model03_PDVI_Config.json
MOVED: Model03_PDVI_Config_v2.json
MOVED: Model03_Second_Test_Prediction.json
MOVED: model03_integrated_result.json
MOVED: model03_predictor.py
MOVED: model03_predictor_v2.py
MOVED previous archive into Model03_Old
REMOVED disposable __pycache__

CLEANUP COMPLETE — VERIFYING MAIN FOLDER
📁 Final_Evaluation_Package_20261020_040237
📁 Integration_Test_20261020_035432
📄 Model03_Estate_Mapping_Audit_20261020_040212.json
📄 Model03_Model02_Feature_List.json
📁 Model03_Old
📁 test_images

Files moved: 9
Old files are preserved in: /content/drive/MyDrive/Tea_Disease_Research/Model03_Outputs/Model03_Old
Model 01/02 folders and the separate Model03 backup were not touched.


In [23]:

import os
import shutil
from google.colab import files

BASE = "/content/drive/MyDrive/Tea_Disease_Research/Model03_Outputs"

# Rename dated folders to clean names
renames = {
    "Final_Evaluation_Package_20261020_040237":
        "Final_Evaluation_Package",
    "Integration_Test_20261020_035432":
        "Integration_Test",
}

for old_name, new_name in renames.items():
    old_path = os.path.join(BASE, old_name)
    new_path = os.path.join(BASE, new_name)

    if os.path.isdir(new_path):
        print("Already exists; left unchanged:", new_name)
    elif os.path.isdir(old_path):
        os.rename(old_path, new_path)
        print("RENAMED:", old_name, "->", new_name)
    else:
        print("Folder not found; skipped:", old_name)

# Rename audit file for a cleaner folder
old_audit = os.path.join(
    BASE, "Model03_Estate_Mapping_Audit_20261020_040212.json"
)
new_audit = os.path.join(
    BASE, "Model03_Estate_Mapping_Audit.json"
)

if os.path.isfile(old_audit) and not os.path.exists(new_audit):
    os.rename(old_audit, new_audit)
    print("Renamed estate mapping audit.")

# Create a clean ZIP of current Model 03 outputs.
# The archived Model03_Old folder is deliberately excluded.
zip_base = "/content/Model03_PDVI_Final"
zip_path = shutil.make_archive(
    zip_base,
    "zip",
    root_dir=BASE,
    # Include only these final files/folders
    base_dir="."
)

# Rebuild ZIP containing only selected final items
import zipfile

include_items = [
    "Final_Evaluation_Package",
    "Integration_Test",
    "test_images",
    "Model03_Estate_Mapping_Audit.json",
    "Model03_Model02_Feature_List.json",
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for item in include_items:
        path = os.path.join(BASE, item)

        if not os.path.exists(path):
            print("Not included (not found):", item)
            continue

        if os.path.isfile(path):
            zf.write(path, arcname=item)
        else:
            for current, dirs, filenames in os.walk(path):
                dirs[:] = [
                    d for d in dirs
                    if d != "__pycache__"
                ]
                for filename in filenames:
                    full_path = os.path.join(current, filename)
                    arcname = os.path.relpath(full_path, BASE)
                    zf.write(full_path, arcname=arcname)

print("\nZIP created:", zip_path)
print("Downloading clean Model 03 output package...")
files.download(zip_path)


Already exists; left unchanged: Final_Evaluation_Package
Already exists; left unchanged: Integration_Test

ZIP created: /content/Model03_PDVI_Final.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>